In [0]:
# 02_preprocess_data

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

catalog = "workspace"
schema = "stroke_prediction"

raw_table = f"{catalog}.{schema}.stroke_raw"
clean_table = f"{catalog}.{schema}.stroke_clean"
train_raw_table = f"{catalog}.{schema}.stroke_train"
test_raw_table = f"{catalog}.{schema}.stroke_test"
train_prepared_table = f"{catalog}.{schema}.stroke_train_prepared"
test_prepared_table = f"{catalog}.{schema}.stroke_test_prepared"
split_summary_table = f"{catalog}.{schema}.stroke_split_summary"

random_state = 42

stroke = spark.table(raw_table).toPandas()

stroke_clean = (
    stroke
    .loc[
        ~stroke["work_type"].isin(["Never_worked", "children"])
        & stroke["gender"].ne("Other")
    ]
    .drop(columns="id")
    .replace({"smoking_status": {"Unknown": np.nan}, "bmi": {"N/A": np.nan}})
    .reset_index(drop=True)
)

if stroke_clean["stroke"].isna().any():
    raise ValueError("Missing stroke outcomes detected.")

if not set(stroke_clean["stroke"].unique()).issubset({0, 1}):
    raise ValueError("Invalid stroke outcomes detected.")

spark.createDataFrame(stroke_clean).write.mode("overwrite").saveAsTable(clean_table)

X = stroke_clean.drop(columns="stroke")
y = stroke_clean["stroke"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=random_state
)

train_raw = X_train.assign(stroke=y_train).reset_index(drop=True)
test_raw = X_test.assign(stroke=y_test).reset_index(drop=True)

spark.createDataFrame(train_raw).write.mode("overwrite").saveAsTable(train_raw_table)
spark.createDataFrame(test_raw).write.mode("overwrite").saveAsTable(test_raw_table)

numeric_columns = [
    "age",
    "bmi",
    "avg_glucose_level"
]

binary_columns = [
    "hypertension",
    "heart_disease"
]

categorical_columns = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("binary", binary_pipeline, binary_columns),
        ("categorical", categorical_pipeline, categorical_columns)
    ],
    verbose_feature_names_out=False
)

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
feature_names = [name.replace(' ', '_') for name in feature_names]

train_prepared = (
    pd.DataFrame(
        X_train_prepared,
        columns=feature_names,
        index=X_train.index
    )
    .assign(stroke=y_train)
    .reset_index(drop=True)
)

test_prepared = (
    pd.DataFrame(
        X_test_prepared,
        columns=feature_names,
        index=X_test.index
    )
    .assign(stroke=y_test)
    .reset_index(drop=True)
)

if train_prepared.isna().any().any():
    raise ValueError("Missing values remain in the prepared training data.")

if test_prepared.isna().any().any():
    raise ValueError("Missing values remain in the prepared test data.")

spark.createDataFrame(train_prepared).write.mode("overwrite").saveAsTable(
    train_prepared_table
)

spark.createDataFrame(test_prepared).write.mode("overwrite").saveAsTable(
    test_prepared_table
)

split_summary = pd.DataFrame({
    "dataset": ["Complete", "Training", "Test"],
    "patients": [
        len(stroke_clean),
        len(train_prepared),
        len(test_prepared)
    ],
    "stroke_cases": [
        int(stroke_clean["stroke"].sum()),
        int(train_prepared["stroke"].sum()),
        int(test_prepared["stroke"].sum())
    ],
    "stroke_prevalence": [
        float(stroke_clean["stroke"].mean()),
        float(train_prepared["stroke"].mean()),
        float(test_prepared["stroke"].mean())
    ]
})

spark.createDataFrame(split_summary).write.mode("overwrite").saveAsTable(
    split_summary_table
)

display(split_summary)